# 🎙️ Douyin Vietnamese Dubbing Server

Extension sẽ tự bấm **Run all**. Nếu tự động không hoạt động, bấm **Runtime → Run all** một lần. Giữ tab này mở trong lúc xử lý video. Notebook dùng GPU Colab, FastAPI và Cloudflare Quick Tunnel tạm thời.

In [ ]:
# Cài mã nguồn và dependency. Cell này có thể chạy lại an toàn.
import hashlib, json, os, pathlib, shutil, subprocess, sys, time
print('NEKO_PROGRESS ' + json.dumps({'stage':'install','progress':20,'message':'Đang cài thư viện AI trên Colab · lần đầu có thể mất vài phút…'}, ensure_ascii=False), flush=True)
ROOT = pathlib.Path('/content/split-video')
REPOSITORY_URL = 'https://github.com/Dattrong0512/split-video.git'
def remove_checkout():
    if ROOT.is_symlink() or ROOT.is_file():
        ROOT.unlink(missing_ok=True)
    elif ROOT.exists():
        shutil.rmtree(ROOT)
def run_git(command):
    return subprocess.run(command, check=True, text=True, capture_output=True)
checkout_error = None
for checkout_attempt in range(1, 4):
    try:
        if (ROOT / '.git').is_dir():
            run_git(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', 'main'])
            run_git(['git', '-C', str(ROOT), 'reset', '--hard', 'origin/main'])
        else:
            remove_checkout()
            run_git(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(ROOT)])
        checkout_error = None
        break
    except subprocess.CalledProcessError as error:
        checkout_error = error
        detail = (error.stderr or error.stdout or str(error)).strip()
        print(f'CHECKOUT_RETRY {checkout_attempt}/3: {detail[-500:]}', flush=True)
        remove_checkout()
        if checkout_attempt < 3:
            time.sleep(2 ** (checkout_attempt - 1))
if checkout_error is not None:
    raise RuntimeError('SOURCE_CHECKOUT_FAILED: ' + (checkout_error.stderr or checkout_error.stdout or str(checkout_error)).strip()) from checkout_error
requirements = ROOT / 'backend/requirements-colab.txt'
noto_font = pathlib.Path('/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf')
if not noto_font.exists():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-noto-core'], check=True)
    subprocess.run(['fc-cache', '-f'], check=True, stdout=subprocess.DEVNULL)
dependency_hash = hashlib.sha256(requirements.read_bytes()).hexdigest()
dependency_marker = pathlib.Path('/content/.douyin-dubbing-dependencies')
if not dependency_marker.exists() or dependency_marker.read_text() != dependency_hash:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', '-r', str(requirements)], check=True)
        dependency_marker.write_text(dependency_hash)
    except Exception as error:
        print('INSTALL_FAILED: ' + str(error), flush=True)
        raise
else:
    print('NEKO_PROGRESS ' + json.dumps({'stage':'cached','progress':48,'message':'Đã dùng lại thư viện AI trong runtime Colab hiện tại.'}, ensure_ascii=False), flush=True)
if not shutil.which('cloudflared'):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb', '-O', '/tmp/cloudflared.deb'], check=True)
    subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], check=True, stdout=subprocess.DEVNULL)
sys.path.insert(0, str(ROOT))
print('NEKO_PROGRESS ' + json.dumps({'stage':'installed','progress':55,'message':'Đã cài thư viện. Đang khởi động máy chủ…'}, ensure_ascii=False), flush=True)

In [ ]:
# Khởi động API và tunnel; extension tự đọc dòng NEKO_SERVER_READY.
import json, os, queue, re, secrets, socket, subprocess, sys, threading, time, urllib.request
print('NEKO_PROGRESS ' + json.dumps({'stage':'gpu','progress':60,'message':'Đã kết nối GPU T4. Đang khởi động API…'}, ensure_ascii=False), flush=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU_UNAVAILABLE: Runtime → Change runtime type → T4 GPU, sau đó Run all lại.')
def stop_process(process):
    if process is None or process.poll() is not None: return
    process.terminate()
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill(); process.wait(timeout=5)
if 'api_process' in globals(): stop_process(api_process)
if 'tunnel' in globals() and tunnel.poll() is None:
    stop_process(tunnel)
with socket.socket() as port_socket:
    port_socket.bind(('127.0.0.1', 0))
    api_port = port_socket.getsockname()[1]
session_token = secrets.token_urlsafe(32)
print('NEKO_PROGRESS ' + json.dumps({'stage':'tunnel','progress':72,'message':'Đang tạo kết nối Cloudflare bảo mật mới…'}, ensure_ascii=False), flush=True)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{api_port}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tunnel_lines = queue.Queue()
def read_tunnel_output():
    for output_line in iter(tunnel.stdout.readline, ''):
        tunnel_lines.put(output_line)
_tunnel_reader = threading.Thread(target=read_tunnel_output, daemon=True)
_tunnel_reader.start()
public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    try: line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        if tunnel.poll() is not None: break
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError('TUNNEL_FAILED: Không tạo được Cloudflare Quick Tunnel.')
print('NEKO_PROGRESS ' + json.dumps({'stage':'api','progress':88,'message':'Đang khởi động worker xử lý mới…'}, ensure_ascii=False), flush=True)
api_env = os.environ.copy()
api_env.update({'DUBBING_SESSION_TOKEN': session_token, 'DUBBING_WORK_ROOT': '/content/douyin-dubbing-jobs', 'DUBBING_PUBLIC_URL': public_url})
api_log_path = '/tmp/douyin-dubbing-api.log'
if 'api_log' in globals() and not api_log.closed: api_log.close()
api_log = open(api_log_path, 'w')
api_process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'backend.server:app', '--host', '127.0.0.1', '--port', str(api_port), '--log-level', 'warning'], cwd=str(ROOT), env=api_env, stdout=api_log, stderr=subprocess.STDOUT, text=True)
api_ready = False
for _ in range(60):
    if api_process.poll() is not None: break
    try:
        ready_request = urllib.request.Request(f'http://127.0.0.1:{api_port}/api/health', headers={'Authorization': 'Bearer ' + session_token})
        urllib.request.urlopen(ready_request, timeout=1)
        api_ready = True
        break
    except Exception:
        time.sleep(1)
if not api_ready:
    api_log.flush()
    detail = pathlib.Path(api_log_path).read_text(errors='replace')[-1000:]
    raise RuntimeError('API_FAILED: Không khởi động được API mới trong Colab. ' + detail)
handshake = {'url': public_url, 'token': session_token, 'createdAt': int(time.time())}
print('NEKO_SERVER_READY ' + json.dumps(handshake, separators=(',', ':')), flush=True)
print('✅ Máy chủ sẵn sàng. Giữ tab này mở; quay lại Douyin để tiếp tục.')
# Thread đọc tunnel log tiếp tục chạy để cloudflared không nghẽn stdout.
# API chạy ở process riêng; Run all lần sau sẽ đóng process cũ và tạo worker sạch.